# 第 07 章练习

完成下面的练习，结合自己的运行结果作答。


<a id="s-07-06_chapter_test"></a>


## 7.6 HCCL 通信与扩展性

先完成单卡数据流和计时分析，有多卡时再比较不同 rank 数。


### 检查设备和工具

下面显示设备信息和 CANN 安装位置，并检查本节要用的命令是否能找到。

In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cmake --version
npu-smi info


### 先选择本次资源

只有一张卡时，执行下面的 `--world-size 1` 检查，并完成数据流与计时分析。获得两张或更多卡后，再做扩展性对照。

单机使用 root-info 初始化即可。需要 rank table 的集群按实验手册查询实际 Device IP。

In [ ]:
%%bash
set -euo pipefail
cd src/hccl_spmv
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DHCCL_SPMV_STUB=OFF
cmake --build build -j
# 单卡实验直接运行；分析其他矩阵时修改 --matrix。
./build/bin/hccl_spmv --rank 0 --world-size 1 --device 0 --matrix U1 --warmup 3 --repeat 10

# 多卡扩展：分配到至少两张卡后再执行。
# bash scripts/run_scaling.sh --matrix U1 --npus-list 1,2 --warmup 3 --repeat 10


### 练习与提交

1. 手算 rows=10、world=4 时各 rank 的行区间与 padding。
2. 保存一次实际运行的后端、设备编号、误差和本地 SpMV 时间。单卡使用 1 rank。
3. 画出 Broadcast → 本地 SpMV → AllGather，指出每轮哪一步搬数据、哪一步计算。解释 synchronization 为什么不能与 launch-to-complete 再相加。
4. 有多卡时比较 1/2 或 2/4 rank；单卡时设计这组实验并列明待测项。计算加速比时使用同参数的 1 rank total，效率再除以 rank 数。

多卡运行时，各 rank 日志保存在程序打印的 `results/run.XXXXXX` 目录中。


[参考答案](answer/07.06_answer.md)
